# 부산 → 인천공항 노선 구간 분할 (v2)

## v1에서 바뀐 것

| # | v1의 문제 | v2의 해결 |
|---|---|---|
| 1 | 도착지 좌표 하드코딩 → 활주로 한복판, `result_code 103` | 지오코딩 함수로 해결. 응답 `result_code` 검사 추가 |
| 2 | 휴게소 목록을 손으로 나열 → **경로에서 최대 72 km 벗어남** | 경로 polyline을 훑으며 카카오 반경검색으로 **발견** |
| 3 | 거리를 도(degree) 유클리드로 계산 후 스칼라 보정 | haversine (미터) |
| 4 | 최근접점 매칭으로 구간거리 근사 | **다중 경유지 API**의 section별 거리·시간 (정확값) |
| 5 | 평균속도가 최종 표에 없음 (ML 입력인데) | section `duration`으로 구간 평균속도 산출 |
| 6 | `ORIGIN` 등 정의 셀 부재 → 커널 재시작 시 붕괴 | 설정 셀 하나로 통합 |
| 7 | 충전소 정보 없음 | 휴게소 ↔ 충전소 반경 매칭 추가 |

## 사전 준비
```bash
pip install requests pandas numpy python-dotenv
```
`.env` 파일:
```
KAKAO_REST_API_KEY=...
DATA_GO_KR_KEY=...        # 환경공단 충전소 API용. 없으면 비워두세요
```

## 0. 설정 및 공통 유틸

In [1]:
import os, time, math, json
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
KAKAO_KEY = os.getenv("KAKAO_REST_API_KEY")
DATA_GO_KR_KEY = os.getenv("DATA_GO_KR_KEY")          # 없으면 None
HEADERS = {"Authorization": f"KakaoAK {KAKAO_KEY}"}

# ── 노선 정의 (다른 노선을 돌리려면 여기만 바꾸면 됩니다) ─────────────
ORIGIN_QUERY = "부산광역시 강서구 녹산산단232로 38-26"
ORIGIN_NAME = "부산 물류센터"
DESTINATION_QUERY = "인천공항 제1여객터미널"
DESTINATION_NAME = "인천국제공항 T1"
PRIORITY = "RECOMMEND"        # or "HIGHWAY_UNIONTOLL" (고속도로 우선)

# ── 탐색 파라미터 ──────────────────────────────────────────────
REST_SEARCH_STEP_KM = 20      # 경로를 몇 km 간격으로 훑을지
REST_SEARCH_RADIUS_M = 3000   # 각 샘플 지점의 검색 반경
REST_MAX_OFFSET_M = 1000      # 경로에서 이만큼 넘게 떨어지면 버림
CHARGER_MATCH_RADIUS_M = 700  # 휴게소 반경 이 안의 충전기를 그 휴게소 것으로 간주
SAME_SIDE_ONLY = True         # 진행방향 오른쪽(상행/하행 일치)만 남길지

# ── 충전소 데이터 소스 ─────────────────────────────────────────
# 한전 CSV는 여러 개를 넣으면 충전소아이디 기준 합집합으로 병합됩니다.
KEPCO_CSVS = [
    "한국전력공사_전기차충전소위경도_20251231.csv",          # 4,673행 (커버리지 넓음)
    "한국전력공사_충전소의 위치 및 현황 정보_20251231.csv",  # 4,394행
]

assert KAKAO_KEY, "KAKAO_REST_API_KEY 가 .env 에 없습니다"
print(f"카카오 키 로드됨: {KAKAO_KEY[:6]}****")
print(f"환경공단 키: {'있음' if DATA_GO_KR_KEY else '없음 (한전 CSV / 카카오로 대체)'}")

카카오 키 로드됨: ed6f28****
환경공단 키: 없음 (한전 CSV / 카카오로 대체)


In [2]:
EARTH_R = 6371008.8  # m


def haversine_m(lng1, lat1, lng2, lat2):
    """두 지점(또는 배열) 사이 대권거리, 미터. 브로드캐스팅 지원."""
    lng1, lat1, lng2, lat2 = (np.asarray(v, dtype=float) for v in (lng1, lat1, lng2, lat2))
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin((p2 - p1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lng2 - lng1) / 2) ** 2)
    return 2 * EARTH_R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def _kakao_get(url, params, retries=3):
    """429(쿼터 초과) 시 백오프 재시도."""
    last = None
    for i in range(retries):
        r = requests.get(url, headers=HEADERS, params=params, timeout=8)
        if r.status_code == 429:
            time.sleep(1.5 * (i + 1))
            last = r
            continue
        r.raise_for_status()
        return r.json()
    last.raise_for_status()


# 간단 자기검증: 서울시청 ↔ 부산시청 실측 약 325 km
_check = haversine_m(126.9780, 37.5665, 129.0756, 35.1796) / 1000
print(f"haversine 검증 (서울-부산 직선): {_check:.1f} km  → 320~330 이면 정상")

haversine 검증 (서울-부산 직선): 325.1 km  → 320~330 이면 정상


## 1. 출발지 / 도착지 좌표

카카오 API는 좌표를 **경도(x), 위도(y)** 순서로 받습니다.

주소로 먼저 시도하고, 실패하면 키워드 검색으로 넘어갑니다. 공항처럼 주소보다 장소명이 자연스러운 곳은
키워드 검색이 잡아줍니다. **좌표를 직접 적어 넣지 마세요** — v1에서 인천공항 좌표를 손으로 넣었다가
활주로 한가운데가 잡혀 `result_code 103 (도착 지점 주변의 도로를 탐색할 수 없음)` 이 났습니다.

In [3]:
def geocode(query):
    """주소 → 실패 시 키워드 검색. 반환: (lng, lat, 확인용 이름)"""
    j = _kakao_get("https://dapi.kakao.com/v2/local/search/address.json", {"query": query})
    if j.get("documents"):
        d = j["documents"][0]
        return float(d["x"]), float(d["y"]), d["address_name"]

    j = _kakao_get("https://dapi.kakao.com/v2/local/search/keyword.json",
                   {"query": query, "size": 1})
    if not j.get("documents"):
        raise ValueError(f"좌표를 찾을 수 없습니다: {query}")
    d = j["documents"][0]
    return float(d["x"]), float(d["y"]), d["place_name"]


ox, oy, o_label = geocode(ORIGIN_QUERY)
dx, dy, d_label = geocode(DESTINATION_QUERY)
ORIGIN_XY, DEST_XY = (ox, oy), (dx, dy)

print(f"출발: {o_label}  ({ox}, {oy})")
print(f"도착: {d_label}  ({dx}, {dy})")
print(f"직선거리: {haversine_m(ox, oy, dx, dy) / 1000:.1f} km")

출발: 부산 강서구 녹산산단232로 38-26  (128.840263337127, 35.0865743070385)
도착: 인천국제공항 제1여객터미널  (126.45240378314084, 37.449691917253595)
직선거리: 338.9 km


## 2. 길찾기 호출

`result_code` 를 반드시 검사합니다. v1은 이걸 안 봐서 실패 응답에 `summary` 키가 없어 엉뚱한 곳에서
`KeyError` 가 났습니다.

In [4]:
NAVI_SINGLE = "https://apis-navi.kakaomobility.com/v1/directions"
NAVI_MULTI = "https://apis-navi.kakaomobility.com/v1/waypoints/directions"


def get_route(origin_xy, dest_xy, waypoints=None, priority=PRIORITY):
    """waypoints 가 있으면 다중 경유지 API(POST), 없으면 단일 길찾기(GET)."""
    if waypoints:
        if len(waypoints) > 30:
            raise ValueError(f"경유지는 최대 30개입니다 (현재 {len(waypoints)}개). "
                             f"REST_SEARCH_STEP_KM 을 늘리거나 급속충전 가능 휴게소만 남기세요.")
        r = requests.post(
            NAVI_MULTI,
            headers={**HEADERS, "Content-Type": "application/json"},
            json={
                "origin": {"x": origin_xy[0], "y": origin_xy[1]},
                "destination": {"x": dest_xy[0], "y": dest_xy[1]},
                "waypoints": [{"name": w["name"], "x": w["lng"], "y": w["lat"]} for w in waypoints],
                "priority": priority,
                "road_details": True,
            },
            timeout=30,
        )
    else:
        r = requests.get(
            NAVI_SINGLE,
            headers=HEADERS,
            params={"origin": f"{origin_xy[0]},{origin_xy[1]}",
                    "destination": f"{dest_xy[0]},{dest_xy[1]}",
                    "priority": priority, "road_details": True},
            timeout=30,
        )
    r.raise_for_status()
    j = r.json()

    route = j["routes"][0]
    code_ = route.get("result_code")
    if code_ != 0:
        raise RuntimeError(f"길찾기 실패 [{code_}] {route.get('result_msg')}\n"
                           f"→ 출발/도착 좌표가 도로에서 너무 멀지 않은지 확인하세요.")
    return j


route_json = get_route(ORIGIN_XY, DEST_XY)
summary = route_json["routes"][0]["summary"]
print(f"총 거리: {summary['distance'] / 1000:.1f} km")
print(f"예상 소요시간: {summary['duration'] / 60:.0f} 분")
print(f"평균속도: {(summary['distance'] / 1000) / (summary['duration'] / 3600):.1f} km/h")

총 거리: 424.7 km
예상 소요시간: 366 분
평균속도: 69.6 km/h


## 3. 경로 polyline 복원 (미터 단위)

v1은 `np.hypot(경도차, 위도차)` 로 도(degree) 공간에서 거리를 재고 마지막에 스칼라 하나로 보정했습니다.
위도 36°에서 경도 1° ≈ 90 km, 위도 1° ≈ 111 km 라 방향에 따라 20% 이상 왜곡됩니다. haversine으로 교체합니다.

In [5]:
def route_polyline(route_json):
    """모든 section/road의 vertexes를 이어 붙여 (lng, lat, cum_km) 테이블 생성."""
    pts = []
    for sec in route_json["routes"][0]["sections"]:
        for road in sec["roads"]:
            v = road["vertexes"]           # [lng1, lat1, lng2, lat2, ...] flat
            pts.extend(zip(v[0::2], v[1::2]))

    df = pd.DataFrame(pts, columns=["lng", "lat"])
    df = df[(df != df.shift()).any(axis=1)].reset_index(drop=True)   # 연속 중복만 제거

    step = haversine_m(df["lng"].values[:-1], df["lat"].values[:-1],
                       df["lng"].values[1:], df["lat"].values[1:])
    df["cum_km"] = np.concatenate([[0.0], np.cumsum(step)]) / 1000
    return df


route_coords = route_polyline(route_json)
api_km = summary["distance"] / 1000
poly_km = route_coords["cum_km"].iat[-1]
print(f"polyline 점 개수: {len(route_coords):,}")
print(f"polyline 누적: {poly_km:.1f} km / API 총거리: {api_km:.1f} km "
      f"(오차 {abs(poly_km - api_km) / api_km * 100:.2f}%)")

polyline 점 개수: 2,755
polyline 누적: 424.7 km / API 총거리: 424.7 km (오차 0.00%)


In [6]:
# 진단: 이 노선이 실제로 어떤 도로를 타는지 확인
roads = []
for sec in route_json["routes"][0]["sections"]:
    for rd in sec["roads"]:
        roads.append({"도로명": rd.get("name") or "무명도로", "거리_m": rd["distance"]})

(pd.DataFrame(roads).groupby("도로명")["거리_m"].sum()
   .sort_values(ascending=False).head(12) / 1000).round(1).rename("거리_km").to_frame()

,거리_km
도로명,
중부내륙고속도로,265.3
영동고속도로,63.5
진해산업로,20.2
인천대교고속도로,18.4
제3경인고속화도로,13.2
수원광명고속도로,11.2
공항로,6.2
봉양로,6.0
아암대로,5.6


## 4. 경로 위의 휴게소 찾기

**여기가 v1의 가장 큰 버그였습니다.** 휴게소 목록을 경부고속도로 기준으로 손으로 적어두었는데,
카카오가 실제로 뽑아준 경로는 다른 길이었습니다. `경로이탈도` 컬럼이 그 증거였죠 —
언양 0.584°(≈58 km), 경주 0.717°(≈72 km), 신탄진 0.658°(≈66 km). **경로 위에 있는 휴게소가 하나도 없었습니다.**

대신 여기서는 polyline을 일정 간격으로 훑으며 반경검색으로 **발견**합니다. 노선이 바뀌어도 자동으로 따라갑니다.

`side` 컬럼은 진행방향 기준 좌/우입니다. 휴게소는 상행·하행이 물리적으로 다른 시설이라
반대편이 같이 잡히는 걸 걸러내기 위한 것입니다 (한국은 우측통행이므로 `+1`이 같은 방향).

In [7]:
def _heading_side(route, idx, lng, lat):
    """경로 진행방향 기준 오른쪽이면 +1, 왼쪽이면 -1."""
    i0, i1 = max(idx - 2, 0), min(idx + 2, len(route) - 1)
    if i0 == i1:
        return 0
    kx = math.cos(math.radians(route["lat"].iat[idx]))          # 경도 축소 보정
    hx = (route["lng"].iat[i1] - route["lng"].iat[i0]) * kx     # 진행 벡터
    hy = route["lat"].iat[i1] - route["lat"].iat[i0]
    px = (lng - route["lng"].iat[idx]) * kx                     # 대상 벡터
    py = lat - route["lat"].iat[idx]
    return -1 if (hx * py - hy * px) > 0 else 1                 # 외적 부호


def find_rest_areas(route, step_km=REST_SEARCH_STEP_KM, radius_m=REST_SEARCH_RADIUS_M,
                    max_offset_m=REST_MAX_OFFSET_M, keyword="휴게소"):
    total = route["cum_km"].iat[-1]
    marks = np.arange(step_km, total, step_km)
    idxs = np.unique(np.searchsorted(route["cum_km"].values, marks))

    found = {}
    for i in idxs:
        p = route.iloc[int(i)]
        j = _kakao_get("https://dapi.kakao.com/v2/local/search/keyword.json",
                       {"query": keyword, "x": p["lng"], "y": p["lat"],
                        "radius": int(radius_m), "sort": "distance", "size": 15})
        for d in j.get("documents", []):
            if keyword not in d["place_name"]:
                continue
            found[d["id"]] = {
                "name": d["place_name"],
                "lng": float(d["x"]), "lat": float(d["y"]),
                "addr": d.get("road_address_name") or d.get("address_name", ""),
            }
        time.sleep(0.05)          # 쿼터 보호

    lngs, lats = route["lng"].values, route["lat"].values
    rows = []
    for r in found.values():
        dist = haversine_m(lngs, lats, r["lng"], r["lat"])
        k = int(dist.argmin())
        if dist[k] > max_offset_m:
            continue
        rows.append({**r,
                     "cum_km": round(float(route["cum_km"].iat[k]), 2),
                     "offset_m": int(round(dist[k])),
                     "side": _heading_side(route, k, r["lng"], r["lat"])})

    if not rows:
        return pd.DataFrame(columns=["name", "lng", "lat", "addr", "cum_km", "offset_m", "side"])
    return (pd.DataFrame(rows).sort_values("cum_km")
            .drop_duplicates("name").reset_index(drop=True))


df_rest = find_rest_areas(route_coords)
print(f"경로상 휴게소 {len(df_rest)}개 발견 (검색 {int(route_coords['cum_km'].iat[-1] // REST_SEARCH_STEP_KM)}지점)")
df_rest

경로상 휴게소 45개 발견 (검색 21지점)


,name,lng,lat,addr,cum_km,offset_m,side
0,워터 영산휴게소 창원방향 전기차충전소,128.495681,35.428659,경남 창녕군 영산면 장척호수길 56-110,59.53,104,-1
1,영산 휴게소 (창원) 전기차충전소,128.495681,35.428659,경남 창녕군 영산면 장척호수길 56-110,59.53,104,-1
2,영산휴게소 주유소 창원방향,128.495722,35.428622,경남 창녕군 영산면 장척호수길 56-110,59.53,100,-1
3,탐앤탐스 영산2호점휴게소 창원방향점,128.495786,35.429836,경남 창녕군 영산면 장척호수길 56-110,59.74,101,-1
4,파스쿠찌 영산휴게소점,128.495714,35.429692,경남 창녕군 영산면 장척호수길 56-110,59.74,102,-1
5,CU E영산휴게소창원점,128.495762,35.429728,경남 창녕군 영산면 장척호수길 56-110,59.74,99,-1
6,공차 영산휴게소 창원방향점,128.495702,35.429753,경남 창녕군 영산면 장척호수길 56-110,59.74,105,-1
7,33떡볶이 영산휴게소 창원방향점,128.495682,35.429733,경남 창녕군 영산면 장척호수길 56-110,59.74,106,-1
8,영산휴게소 창원방향,128.495695,35.429682,경남 창녕군 영산면 장척호수길 56-110,59.74,103,-1
9,101번지남산돈까스 영산휴게소점,128.495674,35.429715,경남 창녕군 영산면 장척호수길 56-110,59.74,106,-1


In [8]:
# 진행방향 필터. side 판정은 진출입로 때문에 흔들릴 수 있으니, 먼저 분포를 보고 켜세요.
print("side 분포:", df_rest["side"].value_counts().to_dict())
print("offset_m 분포:", df_rest["offset_m"].describe()[["min", "50%", "max"]].round(0).to_dict())

if SAME_SIDE_ONLY and (df_rest["side"] == 1).sum() >= 3:
    df_rest = df_rest[df_rest["side"] == 1].reset_index(drop=True)
    print(f"→ 진행방향 우측만 남김: {len(df_rest)}개")
else:
    print("→ 필터 미적용 (SAME_SIDE_ONLY=False 이거나 우측 표본이 너무 적음)")

df_rest

side 분포: {1: 23, -1: 22}
offset_m 분포: {'min': 67.0, '50%': 139.0, 'max': 418.0}
→ 진행방향 우측만 남김: 23개


,name,lng,lat,addr,cum_km,offset_m,side
0,무솔휴게소,128.469592,35.602678,경남 창녕군 대합면 경남대로 5426,81.17,310,1
1,괴산휴게소 양평방향,127.960181,36.831747,충북 괴산군 장연면 중부내륙고속도로 204,242.41,143,1
2,브이시즌 괴산휴게소양평방향점,127.960168,36.831812,충북 괴산군 장연면 중부내륙고속도로 204,242.41,139,1
3,로띠번 괴산휴게소 양평방향점,127.960203,36.831822,충북 괴산군 장연면 중부내륙고속도로 204,242.41,141,1
4,33떡볶이 괴산휴게소 양평방향점,127.960203,36.831822,충북 괴산군 장연면 중부내륙고속도로 204,242.41,141,1
5,엔제리너스 괴산휴게소양평방향점,127.960125,36.831906,충북 괴산군 장연면 중부내륙고속도로 204,242.41,131,1
6,덕평자연휴게소 푸드코트,127.393785,37.241374,경기 이천시 마장면 덕이로154번길 287-76,322.41,153,1
7,벤제프 덕평휴게소점,127.393785,37.241373,경기 이천시 마장면 덕이로154번길 287-76,322.41,153,1
8,담타 덕평휴게소점,127.393780,37.241371,경기 이천시 마장면 덕이로154번길 287-76,322.41,153,1
9,덕평자연휴게소 직원주차장,127.392088,37.243287,경기 이천시 호법면 단천리 1-2,322.41,383,1


## 5. 충전소 데이터 로드

소스가 무엇이든 아래 **공통 스키마**로 흡수합니다:

`station, lng, lat, addr, output_kw, is_fast, source`

### 소스별 특성

| 소스 | 커버리지 | 급속 여부 | 출력(kW) | 키 필요 |
|---|---|---|---|---|
| 한국환경공단 API | 전국 전 사업자 | ✅ | ✅ | data.go.kr |
| 한전 위치정보 CSV | **한전 운영분만 (고속도로 휴게소 35곳)** | ❌ | ❌ | 불필요 |
| 카카오 키워드 검색 | POI 기준, 넓지만 얕음 | ❌ | ❌ | 기존 키 |

첨부하신 한전 CSV는 전국 4,394개 충전소 중 `(고속도로)` 접두어가 붙은 게 **35개**뿐이고,
급속/완속 구분과 충전기 용량 컬럼이 아예 없습니다. 충전 가능 여부의 *하한선* 확인용으로는 쓸 수 있지만,
"이 휴게소에서 몇 kW로 충전할 수 있나"는 답하지 못합니다. 환경공단 API 사용을 권합니다.

In [9]:
STD_COLS = ["station", "lng", "lat", "addr", "output_kw", "is_fast", "source"]


def load_chargers_kepco(csv_paths=KEPCO_CSVS):
    """한전 충전소 CSV 여러 개를 충전소아이디 기준 합집합으로 병합. 인코딩 cp949.
    ※ 이 계열 파일에는 급속/완속 구분과 충전기 용량이 없습니다.
    """
    if isinstance(csv_paths, (str, Path)):
        csv_paths = [csv_paths]

    frames = []
    for p in csv_paths:
        if not Path(p).exists():
            print(f"  건너뜀 (파일 없음): {p}")
            continue
        df = pd.read_csv(p, encoding="cp949")
        df.columns = [c.replace(" ", "") for c in df.columns]   # '충전소 아이디' / '충전소아이디' 혼재
        frames.append(pd.DataFrame({
            "sid": df["충전소아이디"],
            "station": df["충전소명"].astype(str).str.strip(),
            "lng": pd.to_numeric(df["경도"], errors="coerce"),
            "lat": pd.to_numeric(df["위도"], errors="coerce"),
            "addr": df["충전소주소"].astype(str),
            "output_kw": np.nan,      # 이 파일에는 없음
            "is_fast": pd.NA,         # 이 파일에는 없음
            "source": "KEPCO",
        }))
        print(f"  {Path(p).name}: {len(df):,}행")

    if not frames:
        raise FileNotFoundError(f"한전 CSV를 찾을 수 없습니다: {csv_paths}")

    out = (pd.concat(frames, ignore_index=True)
             .dropna(subset=["lng", "lat"])
             .drop_duplicates("sid", keep="first")
             .drop(columns=["sid"]))
    return out[STD_COLS].reset_index(drop=True)


# 환경공단 chgerType: 02(AC완속)만 완속, 나머지는 급속 계열
FAST_TYPES = {"01", "03", "04", "05", "06", "07", "08", "09", "10"}


def load_chargers_keco(service_key, zcodes=None, page_size=9000):
    """한국환경공단 전기자동차 충전소 정보 API.
    zcode(시도코드)별로 페이지네이션. 반경검색이 없어 시도 단위로 받아 로컬 필터링합니다.
    ※ 응답 구조와 시도코드는 data.go.kr 문서에서 한 번 확인하세요 (강원 42→51, 전북 45→52 등 변경 이력 있음).
    """
    url = "http://apis.data.go.kr/B552584/EvCharger/getChargerInfo"
    zcodes = zcodes or ["11", "26", "27", "28", "29", "30", "31", "36",
                        "41", "43", "44", "46", "47", "48", "50", "51", "52"]
    rows = []
    for z in zcodes:
        page = 1
        while True:
            r = requests.get(url, params={"serviceKey": service_key, "pageNo": page,
                                          "numOfRows": page_size, "zcode": z,
                                          "dataType": "JSON"}, timeout=60)
            r.raise_for_status()
            body = r.json()
            items = body.get("items") or body.get("response", {}).get("body", {}).get("items") or {}
            items = items.get("item", []) if isinstance(items, dict) else items
            if not items:
                break
            rows.extend(items)
            if len(items) < page_size:
                break
            page += 1
        print(f"  zcode {z}: 누적 {len(rows):,}건")

    df = pd.DataFrame(rows)
    return pd.DataFrame({
        "station": df["statNm"],
        "lng": pd.to_numeric(df["lng"], errors="coerce"),
        "lat": pd.to_numeric(df["lat"], errors="coerce"),
        "addr": df.get("addr", ""),
        "output_kw": pd.to_numeric(df.get("output"), errors="coerce"),
        "is_fast": df["chgerType"].astype(str).str.zfill(2).isin(FAST_TYPES),
        "source": "KECO",
    }).dropna(subset=["lng", "lat"])[STD_COLS].reset_index(drop=True)


def load_chargers_kakao(df_rest, radius_m=CHARGER_MATCH_RADIUS_M):
    """폴백: 각 휴게소 반경에서 카카오 키워드 검색. 급속 여부·출력은 알 수 없음."""
    rows = []
    for _, ra in df_rest.iterrows():
        j = _kakao_get("https://dapi.kakao.com/v2/local/search/keyword.json",
                       {"query": "전기차충전소", "x": ra["lng"], "y": ra["lat"],
                        "radius": int(radius_m), "sort": "distance", "size": 15})
        for d in j.get("documents", []):
            rows.append({"station": d["place_name"],
                         "lng": float(d["x"]), "lat": float(d["y"]),
                         "addr": d.get("road_address_name", ""),
                         "output_kw": np.nan, "is_fast": pd.NA, "source": "KAKAO"})
        time.sleep(0.05)
    return pd.DataFrame(rows, columns=STD_COLS)


# ── 소스 선택 ──────────────────────────────────────────────
if DATA_GO_KR_KEY:
    print("환경공단 API에서 충전소를 받는 중...")
    df_chargers = load_chargers_keco(DATA_GO_KR_KEY)
elif any(Path(p).exists() for p in KEPCO_CSVS):
    print("한전 CSV 사용 (급속 여부·출력 정보 없음)")
    df_chargers = load_chargers_kepco()
else:
    print("카카오 키워드 검색으로 폴백 (정보량 가장 적음)")
    df_chargers = load_chargers_kakao(df_rest)

print(f"충전소 {len(df_chargers):,}건 로드 / 소스: {df_chargers['source'].iloc[0]}")
df_chargers.head()

카카오 키워드 검색으로 폴백 (정보량 가장 적음)
충전소 112건 로드 / 소스: KAKAO


,station,lng,lat,addr,output_kw,is_fast,source
0,괴산(양평) 휴게소 전기차충전소,127.959996,36.831547,충북 괴산군 장연면 중부내륙고속도로 204,NaN,<NA>,KAKAO
1,괴산(양평) 휴게소 전기차충전소,127.959893,36.831903,충북 괴산군 장연면 중부내륙고속도로 204,NaN,<NA>,KAKAO
2,괴산휴게소 (창원) 전기차충전소,127.957865,36.831409,충북 괴산군 장연면 중부내륙고속도로 205,NaN,<NA>,KAKAO
3,워터 괴산휴게소 창원방향 전기차충전소,127.957550,36.831545,충북 괴산군 장연면 중부내륙고속도로 205,NaN,<NA>,KAKAO
4,괴산(양평) 휴게소 전기차충전소,127.959893,36.831903,충북 괴산군 장연면 중부내륙고속도로 204,NaN,<NA>,KAKAO


## 6. 휴게소 ↔ 충전소 매칭

**주의 — 반대편 휴게소의 충전기가 섞입니다.** 실제로 한전 CSV로 돌려보면 현풍휴게소는 반경 700 m 안에
`(고속도로)현풍휴게소 대구방면` 과 `창원방면` 이 **둘 다** 잡힙니다 (두 시설 간 거리 약 255 m).
고속도로 휴게소는 상·하행이 본선을 사이에 두고 마주 보고 있어서 생기는 문제입니다.

`CHARGER_MATCH_RADIUS_M` 을 200 m 수준으로 줄이면 반대편은 걸러지지만 주차장 끝에 있는 충전기를 놓칠 수 있습니다.
한전 데이터처럼 충전소명에 방면 표기가 있으면 그걸로 거르는 게 정확합니다 (아래 `direction_guard`).

In [10]:
DIRECTION_WORDS = ["방면", "방향", "상행선", "하행선", "상행", "하행", "(상)", "(하)"]


def split_direction(name):
    """'(고속도로)현풍휴게소 대구방면' -> ('(고속도로)현풍휴게소', '대구방면')"""
    for w in DIRECTION_WORDS:
        if w in name:
            head = name.split(w)[0]
            base, _, tag = head.rpartition(" ")
            return (base or head).strip(), (tag + w if base else w)
    return name.strip(), None


def drop_opposite_side(near):
    """같은 휴게소의 반대편 시설만 제거. 같은 방면의 충전기 여러 대는 그대로 둔다."""
    if near.empty:
        return near
    parsed = near["station"].map(split_direction)
    near = near.assign(_base=[p[0] for p in parsed], _dir=[p[1] for p in parsed])

    kept = []
    for _, g in near.groupby("_base", sort=False):
        tagged = g.dropna(subset=["_dir"])
        if tagged["_dir"].nunique() > 1:
            # 방면이 갈리면 휴게소 POI에 더 가까운 쪽만 채택
            best = tagged.groupby("_dir")["_dist_m"].min().idxmin()
            g = g[g["_dir"].isna() | (g["_dir"] == best)]
        kept.append(g)
    return pd.concat(kept).drop(columns=["_base", "_dir"])


def match_chargers(df_rest, df_chargers, radius_m=CHARGER_MATCH_RADIUS_M, use_guard=True):
    lngs, lats = df_chargers["lng"].values, df_chargers["lat"].values
    is_fast = df_chargers["is_fast"]
    rows = []
    for _, ra in df_rest.iterrows():
        d = haversine_m(lngs, lats, ra["lng"], ra["lat"])
        mask = d <= radius_m
        near = df_chargers[mask].assign(_dist_m=d[mask])
        if use_guard:
            near = drop_opposite_side(near)
        fast_n = int((near["is_fast"] == True).sum()) if len(near) else 0
        rows.append({
            **ra.to_dict(),
            "충전기수": len(near),
            "급속기수": fast_n if is_fast.notna().any() else None,
            "최대출력_kW": (float(near["output_kw"].max())
                            if len(near) and near["output_kw"].notna().any() else None),
            "충전소명": " / ".join(sorted(set(near["station"]))[:2]),
        })
    return pd.DataFrame(rows)


df_rest_ev = match_chargers(df_rest, df_chargers)
print(f"충전기 있는 휴게소: {(df_rest_ev['충전기수'] > 0).sum()} / {len(df_rest_ev)}")
df_rest_ev[["name", "cum_km", "offset_m", "충전기수", "급속기수", "최대출력_kW", "충전소명"]]

충전기 있는 휴게소: 22 / 23


,name,cum_km,offset_m,충전기수,급속기수,최대출력_kW,충전소명
0,무솔휴게소,81.17,310,0,None,None,
1,괴산휴게소 양평방향,242.41,143,20,None,None,괴산(양평) 휴게소 전기차충전소 / 괴산휴게소 (창원) 전기차충전소
2,브이시즌 괴산휴게소양평방향점,242.41,139,20,None,None,괴산(양평) 휴게소 전기차충전소 / 괴산휴게소 (창원) 전기차충전소
3,로띠번 괴산휴게소 양평방향점,242.41,141,20,None,None,괴산(양평) 휴게소 전기차충전소 / 괴산휴게소 (창원) 전기차충전소
4,33떡볶이 괴산휴게소 양평방향점,242.41,141,20,None,None,괴산(양평) 휴게소 전기차충전소 / 괴산휴게소 (창원) 전기차충전소
5,엔제리너스 괴산휴게소양평방향점,242.41,131,20,None,None,괴산(양평) 휴게소 전기차충전소 / 괴산휴게소 (창원) 전기차충전소
6,덕평자연휴게소 푸드코트,322.41,153,61,None,None,경기이천 이천프리미엄아울렛 1 전기차충전소 / 덕평자연휴게소(강릉방향) 전기차충전소
7,벤제프 덕평휴게소점,322.41,153,61,None,None,경기이천 이천프리미엄아울렛 1 전기차충전소 / 덕평자연휴게소(강릉방향) 전기차충전소
8,담타 덕평휴게소점,322.41,153,61,None,None,경기이천 이천프리미엄아울렛 1 전기차충전소 / 덕평자연휴게소(강릉방향) 전기차충전소
9,덕평자연휴게소 직원주차장,322.41,383,59,None,None,경기이천 영원무역 이천물류 전기차충전소 / 덕평자연휴게소(강릉방향) 전기차충전소


## 7. 다중 경유지 API로 구간 확정

v1은 휴게소를 polyline 최근접점에 매칭해 누적거리를 **근사**했습니다. 대신 휴게소를 경유지로 넘기면
카카오가 section을 나눠 주고, 각 section의 `distance`/`duration`이 **실측값**으로 옵니다.
평균속도도 여기서 바로 나옵니다.

경유지 상한은 30개, 총 경로 1,500 km 미만입니다.

In [11]:
# 경유지가 30개를 넘으면 충전 가능한 곳 위주로 추립니다.
cand = df_rest_ev.copy()
if len(cand) > 30:
    cand = (cand.sort_values(["충전기수", "offset_m"], ascending=[False, True])
                .head(30).sort_values("cum_km"))
    print(f"경유지 30개 상한 → 충전기 보유 순으로 추림")

waypoints = cand[["name", "lng", "lat"]].to_dict("records")
print(f"경유지 {len(waypoints)}개:", ", ".join(w["name"] for w in waypoints))

경유지 23개: 무솔휴게소, 괴산휴게소 양평방향, 브이시즌 괴산휴게소양평방향점, 로띠번 괴산휴게소 양평방향점, 33떡볶이 괴산휴게소 양평방향점, 엔제리너스 괴산휴게소양평방향점, 덕평자연휴게소 푸드코트, 벤제프 덕평휴게소점, 담타 덕평휴게소점, 덕평자연휴게소 직원주차장, 덕평자연휴게소 국도주차장, 별빛정원 우주테마파크 주차장(덕평휴게소 인천방향), SK에너지수소충전소 덕평휴게소(강릉방향), 덕평휴게소강릉방향 SK에너지수소충전소, 덕평자연휴게소 강릉방향 주차장, 크록스 덕평휴게소점, 브렌우드 덕평휴게소인천방향점, 달콤커피 덕평휴게소 인천방향점, 청담강정 덕평자연휴게소 양방향점, 덕평자연휴게소 인천방향 주차장, 용인휴게소충전소 인천방향, 용인 휴게소 인천방향 수소충전소, 서창산업 용인휴게소주유소 인천방향


In [12]:
def build_segments(origin_xy, dest_xy, waypoints, origin_name, dest_name):
    rj = get_route(origin_xy, dest_xy, waypoints=waypoints)
    names = [origin_name] + [w["name"] for w in waypoints] + [dest_name]
    sections = rj["routes"][0]["sections"]
    assert len(sections) == len(names) - 1, \
        f"section {len(sections)}개 vs 구간 {len(names) - 1}개 — 경유지 순서를 확인하세요"

    rows, cum = [], 0.0
    for i, sec in enumerate(sections):
        km = sec["distance"] / 1000
        hrs = sec["duration"] / 3600
        cum += km
        rows.append({
            "출발": names[i],
            "도착": names[i + 1],
            "구간거리_km": round(km, 2),
            "구간시간_분": round(sec["duration"] / 60, 1),
            "평균속도_kmh": round(km / hrs, 1) if hrs > 0 else np.nan,
            "누적거리_km": round(cum, 2),
        })
    return pd.DataFrame(rows), rj


df_segments, route_wp = build_segments(ORIGIN_XY, DEST_XY, waypoints,
                                       ORIGIN_NAME, DESTINATION_NAME)
df_segments

,출발,도착,구간거리_km,구간시간_분,평균속도_kmh,누적거리_km
0,부산 물류센터,무솔휴게소,78.22,90.0,52.1,78.22
1,무솔휴게소,괴산휴게소 양평방향,162.60,120.6,80.9,240.82
2,괴산휴게소 양평방향,브이시즌 괴산휴게소양평방향점,0.01,0.1,7.2,240.83
3,브이시즌 괴산휴게소양평방향점,로띠번 괴산휴게소 양평방향점,0.00,0.0,NaN,240.83
4,로띠번 괴산휴게소 양평방향점,33떡볶이 괴산휴게소 양평방향점,0.00,0.0,NaN,240.83
5,33떡볶이 괴산휴게소 양평방향점,엔제리너스 괴산휴게소양평방향점,0.01,0.1,8.1,240.84
6,엔제리너스 괴산휴게소양평방향점,덕평자연휴게소 푸드코트,88.67,77.5,68.7,329.51
7,덕평자연휴게소 푸드코트,벤제프 덕평휴게소점,0.00,0.0,NaN,329.51
8,벤제프 덕평휴게소점,담타 덕평휴게소점,0.00,0.0,NaN,329.51
9,담타 덕평휴게소점,덕평자연휴게소 직원주차장,0.24,0.4,35.5,329.75


## 8. 검증

In [13]:
checks = []

# (1) 구간 합 == 총 거리
tot = route_wp["routes"][0]["summary"]["distance"] / 1000
checks.append(("구간거리 합 = 총거리",
               abs(df_segments["구간거리_km"].sum() - tot) < 0.5,
               f"{df_segments['구간거리_km'].sum():.1f} vs {tot:.1f} km"))

# (2) 구간이 비정상적으로 짧지 않은지 (v1은 3.9km, 4.2km 구간이 나왔음)
short = df_segments[df_segments["구간거리_km"] < 5]
checks.append(("5km 미만 구간 없음", len(short) == 0,
               f"{len(short)}개" + (f" ({', '.join(short['도착'])})" if len(short) else "")))

# (3) 평균속도가 현실 범위
spd = df_segments["평균속도_kmh"]
checks.append(("평균속도 20~120 km/h", spd.between(20, 120).all(),
               f"{spd.min():.0f} ~ {spd.max():.0f} km/h"))

# (4) 휴게소 이탈도
checks.append(("모든 휴게소 경로이탈 1km 이내",
               df_rest_ev["offset_m"].max() <= REST_MAX_OFFSET_M,
               f"최대 {df_rest_ev['offset_m'].max()} m"))

# (5) 경유지 순서 == 누적거리 순서
checks.append(("경유지가 경로 순서대로 정렬됨",
               df_segments["누적거리_km"].is_monotonic_increasing, "—"))

for name, ok, detail in checks:
    print(f"[{'OK ' if ok else 'FAIL'}] {name}: {detail}")

[OK ] 구간거리 합 = 총거리: 563.8 vs 563.8 km
[FAIL] 5km 미만 구간 없음: 12개 (브이시즌 괴산휴게소양평방향점, 로띠번 괴산휴게소 양평방향점, 33떡볶이 괴산휴게소 양평방향점, 엔제리너스 괴산휴게소양평방향점, 벤제프 덕평휴게소점, 담타 덕평휴게소점, 덕평자연휴게소 직원주차장, 덕평자연휴게소 국도주차장, 덕평휴게소강릉방향 SK에너지수소충전소, 브렌우드 덕평휴게소인천방향점, 달콤커피 덕평휴게소 인천방향점, 청담강정 덕평자연휴게소 양방향점)
[FAIL] 평균속도 20~120 km/h: 4 ~ 81 km/h
[OK ] 모든 휴게소 경로이탈 1km 이내: 최대 418 m
[OK ] 경유지가 경로 순서대로 정렬됨: —


## 9. 저장

In [14]:
OUT_SEG = "busan_incheon_route_segments.csv"
OUT_REST = "busan_incheon_rest_areas_ev.csv"

df_segments.to_csv(OUT_SEG, index=False, encoding="utf-8-sig")
df_rest_ev.to_csv(OUT_REST, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUT_SEG} ({len(df_segments)}행), {OUT_REST} ({len(df_rest_ev)}행)")

저장 완료: busan_incheon_route_segments.csv (24행), busan_incheon_rest_areas_ev.csv (23행)


---

## 다음 노트북(ML)으로 넘어갈 때

`df_segments` 가 X의 뼈대입니다. 현재 컬럼: `구간거리_km`, `구간시간_분`, `평균속도_kmh`.

kWh/100km 회귀에 추가로 넣으면 좋을 것들:

- **고도차** — EV 전비에 가장 크게 작용합니다. 카카오에는 고도 정보가 없으니 국토지리정보원 수치표고모델(DEM)이나
  Open-Elevation으로 polyline 각 점의 표고를 받아 구간별 상승/하강 누적을 계산하세요. 회생제동 때문에
  단순 고도차가 아니라 **상승분과 하강분을 따로** 넣는 게 낫습니다.
- **도로 등급 비율** — 각 section의 `roads[].name` 으로 고속도로/국도 비율을 계산할 수 있습니다.
- **적재중량** — 물류 운송이면 이게 거리 다음으로 큽니다.
- **기온** — 배터리 효율과 공조 부하. 기상청 ASOS 시간자료.

첨부하신 `한국전력공사_전기차 시간대별 충전부하` 는 전국 집계라 구간 전비 예측에는 못 씁니다.
다만 **충전 대기시간**을 별도로 모델링한다면 시간대 feature로 쓸 수 있습니다.